In [ ]:
# Import ---------------------------------------------------

import argparse
import os
import sys

import pickle
import itertools
import datetime
import numpy as np

from tqdm import tqdm

import albumentations as trans
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from pathlib import Path
_repo_name = "Example_Use"

current_path = str(Path.cwd())
_module_path = current_path[:current_path.find(_repo_name) + len(_repo_name)]
sys.path.append(os.path.abspath(_module_path))

from utils.functions import set_random, load_config
from src.data_loading import SpectralImagesUnsupervised, load_labels, get_dataset_path
from src.models import Encoder, Decoder


from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


set_random(42)

config_path = os.path.join(_module_path, "src", "config.yml")
config = load_config(config_path)

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device = ', device)

In [ ]:
# ------------------------------------------------
# Batch size for train and inference 
# ------------------------------------------------
train_batch_size = config["hyperparameters"]["train_batch_size"]
inference_batch_size = config["hyperparameters"]["inference_batch_size"]

# ------------------------------------------------
# Image Transformations 
# ------------------------------------------------

base_transform = trans.Compose(
    [
        ToTensorV2(),
    ]
)

train_transform = trans.Compose(
    [
        #trans.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        base_transform,
    ]
)

In [ ]:
# ------------------------------------------------
# Data Loading 
# ------------------------------------------------

def get_dataset(root, df, load_dt_images=True):

    train_samples = df[df["split_unsupervised"] == "train"]
    trainset = SpectralImagesUnsupervised(
        root,
        transform=train_transform,
        dataframe=train_samples,
        load_dt_images=load_dt_images,
        data_type="train",
    )

    valid_samples = df[df["split_unsupervised"] == "valid"]
    validset = SpectralImagesUnsupervised(
        root,
        transform=base_transform,
        dataframe=valid_samples,
        load_dt_images=load_dt_images,
        data_type="valid",
        max = trainset.max,
        min = trainset.min,
    )

    test_samples = df[df["split_unsupervised"] == "test"]
    testset = SpectralImagesUnsupervised(
        root,
        transform=base_transform,
        dataframe=test_samples,
        load_dt_images=load_dt_images,
        data_type="test",
        max = trainset.max,
        min = trainset.min,
    )

    trainloader = DataLoader(
        trainset,
        batch_size=512,
        shuffle=True,
        pin_memory=True,
        drop_last=True,
    )

    validloader = DataLoader(
        validset,
        batch_size=inference_batch_size,
        shuffle=False,
        pin_memory=True,
        drop_last=False,
    )

    testloader = DataLoader(
        testset,
        batch_size=inference_batch_size,
        shuffle=False,
        pin_memory=True,
        drop_last=False,
    )

    return trainloader, validloader, testloader


In [ ]:
# NOTE: the one-class SVM results were obtained on dataset 2 and have not been re-run on the
# current dataset yet. The trained SVMs (Weights/OUT/OneClassSVM_*) are not part of the repository.
data_path = get_dataset_path(config["dataset_nr"])
df = load_labels(data_path)
trainloader, validloader, testloader = get_dataset(data_path, df, load_dt_images=False)

## ROC Functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 9
plt.rcParams["pdf.fonttype"] = 42  # embed TrueType instead of Type 3 fonts



def ROC_plot(true_classes, anomaly_score):

    jammer_list = ['no jammer', 'barrage', 'deceptive', 'pilot', 'sweep', 'random_hop']
    true_label = []
    for l in true_classes:
        true_label.append(jammer_list[int(l)])

    true_label = np.array(true_label)

    # All anomaly types (exclude normal label 0)
    #anomaly_labels = np.unique(true_label)
    #anomaly_labels = anomaly_labels[anomaly_labels != 'no jammer']
    anomaly_labels = ['barrage', 'deceptive', 'pilot', 'sweep', 'random_hop']

    fig, ax = plt.subplots(figsize=(3.0, 2.6), layout="constrained")

    for label in anomaly_labels:
        # Compare this anomaly type against normal samples only.
        # Other anomaly types are excluded from this ROC curve.
        mask = (true_label == 'no jammer') | (true_label == label)

        y_true = (true_label[mask] == label).astype(int)
        y_score = anomaly_score[mask]

        fpr, tpr, thresholds = roc_curve(y_true, y_score)
        auc = roc_auc_score(y_true, y_score)

        ax.plot(
            fpr,
            tpr,
            linewidth=2,
            label=f"{label} ({auc:.2f})"
        )

    # Random classifier
    ax.plot(
        [0, 1],
        [0, 1],
        ls="--", 
        lw=0.7, 
        color="0.6"
        )
    ax.grid(True, lw=0.4, color="0.85")
    ax.set(
        xlim=(0, 1), ylim=(0, 1.01),
        xlabel="False positive rate", 
        ylabel="True positive rate"
        )
    ax.legend(loc="lower right")
    #plt.xlim([0, 1])
    #plt.ylim([0, 1.05])
    #plt.grid(alpha=0.3)
    #plt.legend()
    #plt.tight_layout()
    
    plt.savefig(os.path.join("..", "Figures", "OCSVM.pdf"))
    plt.show()

In [ ]:
def get_scores(model, svm, scaler, testloader):

    test_features = []
    test_labels = []

    with torch.inference_mode():

        for id_t, img, ab, _, _ in tqdm(
            testloader,
            desc="Extracting features (testset)"
        ):
            assert all(x[:6] == id_t[0][:6] for x in id_t), "Sensing units are from different sampels"

            img = img.float().to(device)

            features = model(img)

            features = features.flatten(
                start_dim=1
            )

            test_features.append(
                features.cpu().numpy()
            )

            test_labels.append(
                ab.cpu().numpy()
            )

    anomaly_score = []
    true_label = []

    for x_test, label in tqdm(zip(test_features, test_labels), desc="score calculation"):
        X_test_scaled = scaler.transform(
            x_test
            )
        svm_distance = np.min(svm.decision_function(X_test_scaled))
        anomaly_score.append(-svm_distance)
        true_label.append(label[0])

    true_label = np.asarray(true_label)
    anomaly_score = np.asarray(anomaly_score)

    
    #ROC_plot(true_label, anomaly_score)

    return true_label, anomaly_score

## Encoder Fine-Tuning

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights


class ResNet18FE(nn.Module):
    def __init__(self, pretrained=True, freeze_backbone=False):
        super().__init__()

        # Load pretrained ResNet18
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        self.backbone = resnet18(weights=weights)

        # ------------------------------------------------
        # Change input from 3 channels -> 1 channel
        # ------------------------------------------------
        old_conv = self.backbone.conv1

        self.backbone.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False,
        )

        # Initialize 1-channel convolution from pretrained RGB weights
        if pretrained:
            with torch.no_grad():
                self.backbone.conv1.weight.copy_(
                    old_conv.weight.mean(dim=1, keepdim=True)
                )

        # ------------------------------------------------
        # Remove ImageNet classification layer
        # ------------------------------------------------
        self.backbone.fc = nn.Identity()

        # ------------------------------------------------
        # Optional: freeze backbone
        # ------------------------------------------------
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

    def forward(self, x):
        return self.backbone(x)

In [ ]:
# ---------------------------------------------------------
# Projection head
# ---------------------------------------------------------
class ProjectionHead(nn.Module):
    def __init__(self, in_dim=512, hidden_dim=256, out_dim=128):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)



In [ ]:
import torch.nn.functional as F
# ---------------------------------------------------------
# NT-Xent / InfoNCE loss
# ---------------------------------------------------------
def contrastive_loss(z1, z2, temperature=0.5):

    # Normalize embeddings
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    batch_size = z1.size(0)

    # Concatenate both views
    z = torch.cat([z1, z2], dim=0)       # [2B, D]

    # Similarity matrix
    similarity = torch.matmul(z, z.T)    # [2B, 2B]

    # Remove self-similarity
    mask = torch.eye(
        2 * batch_size,
        device=z.device,
        dtype=torch.bool
    )

    similarity = similarity / temperature
    similarity.masked_fill_(mask, -float("inf"))

    # Positive pair for each sample:
    # i -> i+B
    # i+B -> i
    labels = torch.cat([
        torch.arange(batch_size, 2 * batch_size),
        torch.arange(0, batch_size)
    ]).to(z.device)

    loss = F.cross_entropy(similarity, labels)

    return loss

In [ ]:
# ---------------------------------------------------------
# Data augmentation
# ---------------------------------------------------------
import torchvision.transforms as T

contrastive_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomApply([
        T.ColorJitter(
            brightness=0.4,
            contrast=0.4,
            saturation=0.4,
            hue=0.1,
        )
    ], p=0.8),
    T.RandomGrayscale(p=0.2),
])


In [ ]:

def train(model, projection_head, trainloader, validloader, optimizer, scheduler, num_epochs):

    trial_name = f"Tuned_ResNet18_{now}"

    out_dir = os.path.join(
        _module_path,
        "Weights",
        "OUT",
        trial_name
    )

    os.makedirs(out_dir, exist_ok=True)

    best_valid_loss = float("inf")

    for epoch in range(num_epochs):

        # -------------------------------------------------
        # Training
        # -------------------------------------------------

        model.train()
        projection_head.train()

        running_loss = 0.0

        for _, img, _, _, _ in trainloader:

            img = img.float()
            img = img.to(device)

            # ---------------------------------------------
            # Create two different views of each image
            # ---------------------------------------------
            img1 = torch.stack([
                contrastive_transform(x)
                for x in img
            ]).to(device)

            img2 = torch.stack([
                contrastive_transform(x)
                for x in img
            ]).to(device)

            optimizer.zero_grad()

            # ---------------------------------------------
            # ResNet embeddings
            # ---------------------------------------------
            h1 = model(img1)
            h2 = model(img2)

            # ---------------------------------------------
            # Projection head
            # ---------------------------------------------
            z1 = projection_head(h1)
            z2 = projection_head(h2)

            # ---------------------------------------------
            # Contrastive loss
            # ---------------------------------------------
            loss = contrastive_loss(
                z1,
                z2,
                temperature=0.5
            )

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(trainloader)

        scheduler.step()


        # -------------------------------------------------
        # Validation
        # -------------------------------------------------

        model.eval()
        projection_head.eval()

        valid_loss = 0.0

        with torch.no_grad():

            for _, img, _, _, _ in validloader:

                img = img.float()
                img = img.to(device)

                img1 = torch.stack([
                    contrastive_transform(x)
                    for x in img
                ]).to(device)

                img2 = torch.stack([
                    contrastive_transform(x)
                    for x in img
                ]).to(device)

                h1 = model(img1)
                h2 = model(img2)

                z1 = projection_head(h1)
                z2 = projection_head(h2)

                loss = contrastive_loss(
                    z1,
                    z2,
                    temperature=0.5
                )

                valid_loss += loss.item()

        valid_loss /= len(validloader)


        # -------------------------------------------------
        # Logging
        # -------------------------------------------------

        print(
            f"Epoch [{epoch + 1}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f} "
            f"Valid Loss: {valid_loss:.4f}"
        )


        # -------------------------------------------------
        # Save best model
        # -------------------------------------------------

        if valid_loss < best_valid_loss:

            best_valid_loss = valid_loss

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "projection_head_state_dict":
                        projection_head.state_dict(),
                    "optimizer_state_dict":
                        optimizer.state_dict(),
                    "train_loss": train_loss,
                    "valid_loss": valid_loss,
                },
                os.path.join(
                    out_dir,
                    "best_model.pth"
                )
            )


    return model, projection_head

In [ ]:
# ---------------------------------------------------------
# Training
# ---------------------------------------------------------
FT_model = ResNet18FE(
    pretrained=True,
    freeze_backbone=False,
).to(device)

projection_head = ProjectionHead(
    in_dim=512,
    hidden_dim=256,
    out_dim=128,
).to(device)

now = datetime.datetime.now().strftime("%m%d-%H%M%S")

num_epochs = 100

In [ ]:
optimizer = torch.optim.AdamW(
    list(FT_model.parameters()) +
    list(projection_head.parameters()),
    lr=1e-2,
    weight_decay=1e-2,
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=30,
    gamma=0.7,
)

train(FT_model, projection_head, trainloader, validloader, optimizer, scheduler, num_epochs)

## SVM Train

### SVM

In [ ]:
# ------------------------------------------------
# Train One-Class SVM
# ------------------------------------------------

def SVM_train(
    now,
    device,
    model,
    trainloader,
    nu=0.05,
    gamma="scale",
):

    try:

        trial_name = f"OneClassSVM_{now}"

        out_dir = os.path.join(
            _module_path,
            "Weights",
            "OUT",
            trial_name
        )
        os.makedirs(out_dir, exist_ok=True)

        # ------------------------------------------------
        # 1. Extract NORMAL training features
        # ------------------------------------------------

        model.eval()

        train_features = []

        with torch.inference_mode():

            for _, orig_img, _, _, _ in tqdm(
                trainloader,
                desc="Extracting training features"
            ):

                orig_img = orig_img.float().to(device)

                # Feature extraction
                features = model(orig_img)

                # Flatten if model output is not already
                # [batch_size, n_features]
                features = features.flatten(start_dim=1)

                train_features.append(
                    features.cpu().numpy()
                )


        # Combine all batches
        X_train = np.concatenate(
            train_features,
            axis=0
        )

        print()
        print("Training feature shape:", X_train.shape)


        # ------------------------------------------------
        # 2. Standardize features
        # ------------------------------------------------

        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(
            X_train
        )


        # ------------------------------------------------
        # 3. Train One-Class SVM
        # ------------------------------------------------

        print("Train One-Class SVM ...")

        svm = OneClassSVM(
            kernel="rbf",
            nu=nu,
            gamma=gamma
        )

        svm.fit(X_train_scaled)

        print("One-Class SVM training finished.")

        # ------------------------------------------------
        # Save SVM + scaler
        # ------------------------------------------------

        with open(
            os.path.join(
                out_dir,
                "one_class_svm.pkl"
            ),
            "wb"
        ) as f:

            pickle.dump(svm, f)


        with open(
            os.path.join(
                out_dir,
                "feature_scaler.pkl"
            ),
            "wb"
        ) as f:

            pickle.dump(scaler, f)


        print()
        print("Models saved to:")
        print(out_dir)

        #return svm, scaler, anomaly_score, true_label


    except Exception as e:

        print(f"An error occurred: {e}")

        raise

### Pre-trained ResNet18

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights


class ResNet18FE(nn.Module):
    def __init__(self, pretrained=True, freeze_backbone=False):
        super().__init__()

        # Load pretrained ResNet18
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        self.backbone = resnet18(weights=weights)

        # ------------------------------------------------
        # Change input from 3 channels -> 1 channel
        # ------------------------------------------------
        old_conv = self.backbone.conv1

        self.backbone.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False,
        )

        # Initialize 1-channel convolution from pretrained RGB weights
        if pretrained:
            with torch.no_grad():
                self.backbone.conv1.weight.copy_(
                    old_conv.weight.mean(dim=1, keepdim=True)
                )

        # ------------------------------------------------
        # Remove ImageNet classification layer
        # ------------------------------------------------
        self.backbone.fc = nn.Identity()

        # ------------------------------------------------
        # Optional: freeze backbone
        # ------------------------------------------------
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

    def forward(self, x):
        return self.backbone(x)

In [ ]:
model = ResNet18FE(
    pretrained=True,
    freeze_backbone=True,
).to(device)

now = datetime.datetime.now().strftime("%m%d-%H%M%S")
now

In [ ]:
SVM_train(
    now,
    device,
    model,
    trainloader,
    nu=0.05,
    gamma="scale",
)

In [ ]:
trial_name = f"OneClassSVM_{now}"

out_dir = os.path.join(
    _module_path,
    "Weights",
    "OUT",
    trial_name
)

model_dir = out_dir

# Load SVM model
with open(os.path.join(model_dir, "one_class_svm.pkl"), "rb") as f:
    svm = pickle.load(f)

# Load feature scaler
with open(os.path.join(model_dir, "feature_scaler.pkl"), "rb") as f:
    scaler = pickle.load(f)

In [ ]:
print("Feature Extraction Model: Pretrained ResNet18")
true_classes, anomaly_score = get_scores(model, svm, scaler, testloader)

In [ ]:
ROC_plot(true_classes, anomaly_score)

### Fine-tuned ResNet18

In [ ]:
model = ResNet18FE(
    pretrained=True,
    freeze_backbone=True,
).to(device)

model.load_state_dict(
    # NOTE: these fine-tuned ResNet18 weights are not part of the repository
    torch.load(os.path.join("..", "Weights", "OUT", "Tuned_ResNet18_1008-010148", "best_model.pth"), map_location=device, weights_only=True),
    strict = False
)
now = datetime.datetime.now().strftime("%m%d-%H%M%S")
now

In [ ]:
SVM_train(
    now,
    device,
    model,
    trainloader,
    nu=0.05,
    gamma="scale",
)

In [ ]:
trial_name = f"OneClassSVM_{now}"

out_dir = os.path.join(
    _module_path,
    "Weights",
    "OUT",
    trial_name
)

model_dir = out_dir

# Load SVM model
with open(os.path.join(model_dir, "one_class_svm.pkl"), "rb") as f:
    svm = pickle.load(f)

# Load feature scaler
with open(os.path.join(model_dir, "feature_scaler.pkl"), "rb") as f:
    scaler = pickle.load(f)

print("Feature Extraction Model: Pretrained ResNet18")

get_scores(model, svm, scaler, testloader)